## Configuration

Set up the working environment:

In [ ]:
# Load basic libraries
import os
import sys
from pathlib import Path

# Are we working on google colab?
if 'google.colab' in str(get_ipython()):
    
    from google.colab import drive
    drive.mount('/content/drive')
    
    # If so, look for the files on the drive location
    data_path = Path('/content/drive/MyDrive/TFM/data')
    
else:
    
    # If the session is running local
    root_path = Path(os.path.abspath(".."))
    
    # Add path to Python's system
    if str(root_path) not in sys.path:
        sys.path.append(str(root_path))
    
    # Create file path
    file_path = root_path / 'data' / 'intermediate' / 'clinical_data_clean.parquet'

Load libraries:

In [ ]:
# DATA WRANGLING AND STATISTICS
import pandas as pd
import numpy as np
from scipy.stats import randint, uniform, loguniform

# DATA PREPROCESSING
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# HYPERPARAMETER TUNING
import optuna
from optuna.integration import OptunaSearchCV
from optuna.distributions import IntDistribution, FloatDistribution, CategoricalDistribution
optuna.logging.set_verbosity(optuna.logging.WARNING)

# MODEL SELECTION
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (RandomForestClassifier, 
                            ExtraTreesClassifier, 
                            AdaBoostClassifier, 
                            GradientBoostingClassifier)

from sklearn.neural_network import MLPClassifier

# MODEL TRAINING 
from src.models.training import optimize_model

# MODEL EVALUATION
from src.models.plots import plot_cm



## Data

Load the data, check the structure and split the data into target class and features:

In [ ]:
# Load data
df = pd.read_parquet(file_path)

# Check general structure
df.info()

# Features
X = df.drop("af_recurrence", axis=1)

# Target class manually encoded
y = df["af_recurrence"].map({"no":0, "yes":1})

Since the `cohort` and `node` features are meant to randomize the subjects to perform statistical analysis, it does not make any sense to use them as predictors. Therefore, they can be dropped:

In [ ]:
# Drop features
X = X.drop(['cohort', 'node'], axis=1)

## Preprocessing

Divide data set into train and test set:

In [ ]:
# Divide into train and test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=y
    )

Create the preprocessing pipeline:

In [ ]:
# Divide features by their data type
numeric_features = X.select_dtypes(include=[np.number]).columns
categorical_features = X.select_dtypes(include='category').columns

# Transformers
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('cat', OneHotEncoder(handle_unknown='ignore'))
])

# Join transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

## Training and optimization

Training each model with stratified 5-fold cross validation:

In [ ]:
my_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

### Random forest

Set up the pipeline and the parameter distribution:

In [ ]:
# Full pipeline
pipe_RF = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier(random_state=42))
])

# Hyperparameters search space
params_dist_RF = {
    'clf__n_estimators': IntDistribution(50, 300),
    'clf__max_depth': IntDistribution(2, 32),
    'clf__min_samples_split': IntDistribution(2, 20), 
    'clf__criterion': CategoricalDistribution(['gini', 'entropy']),
    'clf__class_weight': CategoricalDistribution([None, 'balanced', 'balanced_subsample'])
}

Train and optimize the model:

In [ ]:
(optimized_RF, 
test_metrics_RF, 
val_metrics_mean_RF, 
val_metrics_std_RF, 
precs_RF, 
recs_RF) = optimize_model(pipe_RF, params_dist_RF, 
                        X_train, y_train, X_test, y_test,
                        cv=my_cv)

Check the confusion matrix:

In [ ]:
# Compute predictions for the test set
y_pred_RF = optimized_RF.predict(X_test)

# Display confusion matrix
plot_cm(y_true=y_test, y_pred=y_pred_RF, 
        title="Random Forest - Test Set")

### Support Vector Machine


Set up the pipeline and the parameter distributions:

In [ ]:
# Full pipeline
pipe_SVM = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', SVC(random_state=42))
])

# Parameters grid
params_dist_SVM = {
    'clf__C': FloatDistribution(1e-3, 1e2, log=True),
    'clf__kernel': CategoricalDistribution(['linear', 'rbf', 'poly', 'sigmoid']),
    'clf__gamma': CategoricalDistribution(['scale', 'auto']), 
    'clf__degree': IntDistribution(2, 5),
    'clf__class_weight': CategoricalDistribution([None, 'balanced'])
}

Train and optimize the model:

In [ ]:
(optimized_SVM, 
test_metrics_SVM, 
val_metrics_mean_SVM, 
val_metrics_std_SVM, 
precs_SVM, 
recs_SVM) = optimize_model(pipe_SVM, params_dist_SVM, 
                        X_train, y_train, X_test, y_test,
                        cv=my_cv)

Check the confusion matrix:

In [ ]:
# Compute predictions for the test set
y_pred_SVM = optimized_SVM.predict(X_test)

# Display confusion matrix
plot_cm(y_true=y_test, y_pred=y_pred_SVM, 
        title="SVM - Test Set")

### K-Nearest Neighbors

Set up the pipeline and the parameters distributions:

In [ ]:
# Full pipeline
pipe_KNN = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', KNeighborsClassifier())
])

# Parameters grid
params_dist_KNN = {
    'clf__n_neighbors': IntDistribution(3, 50),
    'clf__weights': CategoricalDistribution(['uniform', 'distance']),
    'clf__p': IntDistribution(1, 2),
    'clf__algorithm': CategoricalDistribution(['auto', 'ball_tree', 'kd_tree', 'brute'])
}

Train and optimize the model:

In [ ]:
(optimized_KNN, 
test_metrics_KNN, 
val_metrics_mean_KNN, 
val_metrics_std_KNN, 
precs_KNN, 
recs_KNN) = optimize_model(pipe_KNN, params_dist_KNN, 
                        X_train, y_train, X_test, y_test,
                        cv=my_cv)

Check the confusion matrix:

In [ ]:
# Compute predictions for the test set
y_pred_SVM = optimized_SVM.predict(X_test)

# Display confusion matrix
plot_cm(y_true=y_test, y_pred=y_pred_SVM, 
        title="SVM - Test Set")

## Comparison between models

Evaluate the precision-recall curve: